In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv('Weather_dataset.csv')
print("Data shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData info:")
print(df.info())

# Data preprocessing
def preprocess_weather_data(df):
    # Convert time column to datetime
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

    # Handle missing values
    df = df.interpolate(method='time')

    # Convert wind direction to sine and cosine components
    df['wind_direction_sin'] = np.sin(np.deg2rad(df['wind_direction']))
    df['wind_direction_cos'] = np.cos(np.deg2rad(df['wind_direction']))
    df = df.drop('wind_direction', axis=1)

    return df

# Prepare sequences for RNN
def prepare_multivariate_sequences(data, seq_length, target_col='temperature'):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data.iloc[i:(i + seq_length)].values)
        y.append(data[target_col].iloc[i + seq_length])
    return np.array(X), np.array(y)

# Create and train RNN model
def create_lstm_model(input_shape, dropout_rate=0.2):
    model = Sequential([
        LSTM(100, activation='relu', input_shape=input_shape, return_sequences=True),
        Dropout(dropout_rate),
        LSTM(50, activation='relu'),
        Dropout(dropout_rate),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train and evaluate models
def train_and_evaluate_models(df, target_col='temperature', seq_length=24):
    # Split data into train and test sets (80-20 split)
    train_size = int(len(df) * 0.8)
    train_data = df[:train_size]
    test_data = df[train_size:]

    # Scale the data
    scaler_dict = {}
    scaled_data = pd.DataFrame()

    for column in df.columns:
        scaler = MinMaxScaler()
        scaled_data[column] = scaler.fit_transform(df[column].values.reshape(-1, 1)).flatten()
        scaler_dict[column] = scaler

    # Prepare data for RNN
    X_train, y_train = prepare_multivariate_sequences(
        scaled_data[:train_size],
        seq_length,
        target_col
    )
    X_test, y_test = prepare_multivariate_sequences(
        scaled_data[train_size:],
        seq_length,
        target_col
    )

    # Train RNN
    lstm_model = create_lstm_model((seq_length, X_train.shape[2]))
    lstm_history = lstm_model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.1,
        verbose=1
    )

    # Make RNN predictions
    lstm_predictions_scaled = lstm_model.predict(X_test)
    lstm_predictions = scaler_dict[target_col].inverse_transform(
        lstm_predictions_scaled.reshape(-1, 1)
    )

    # Train SARIMA model
    # Using only the target variable for SARIMA
    target_series = df[target_col][:train_size]
    sarima_model = SARIMAX(
        target_series,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 24)  # Assuming hourly data
    )
    sarima_fitted = sarima_model.fit(disp=False)

    # Make SARIMA predictions
    sarima_predictions = sarima_fitted.forecast(len(test_data) - seq_length)

    # Get actual values
    actual_values = scaler_dict[target_col].inverse_transform(
        y_test.reshape(-1, 1)
    )

    # Calculate metrics
    def calculate_all_metrics(actual, predicted, model_name):
        mse = mean_squared_error(actual, predicted)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(actual, predicted)
        r2 = r2_score(actual, predicted)

        return pd.DataFrame({
            'Model': [model_name],
            'MSE': [mse],
            'RMSE': [rmse],
            'MAE': [mae],
            'R2': [r2]
        })

    # Compare models
    lstm_metrics = calculate_all_metrics(actual_values, lstm_predictions, 'LSTM')
    sarima_metrics = calculate_all_metrics(actual_values, sarima_predictions, 'SARIMA')

    # Plot results
    plt.figure(figsize=(15, 6))
    plt.plot(actual_values, label='Actual', alpha=0.7)
    plt.plot(lstm_predictions, label='LSTM Predictions', alpha=0.7)
    plt.plot(sarima_predictions, label='SARIMA Predictions', alpha=0.7)
    plt.title(f'Temperature Prediction: LSTM vs SARIMA')
    plt.xlabel('Time Steps')
    plt.ylabel('Temperature (°C)')
    plt.legend()
    plt.grid(True)

    return {
        'metrics': pd.concat([lstm_metrics, sarima_metrics]),
        'lstm_history': lstm_history,
        'predictions': {
            'actual': actual_values,
            'lstm': lstm_predictions,
            'sarima': sarima_predictions
        }
    }

# Main execution
def main(df):
    # Print initial data information
    print("\nInitial data exploration:")
    print("Checking for missing values:")
    print(df.isnull().sum())

    # Preprocess data
    print("\nPreprocessing data...")
    processed_df = preprocess_weather_data(df)

    print("\nProcessed data shape:", processed_df.shape)
    print("\nProcessed columns:", processed_df.columns.tolist())

    # Train and evaluate models
    print("\nTraining models...")
    results = train_and_evaluate_models(processed_df)

    # Print results
    print("\nModel Comparison Metrics:")
    print(results['metrics'])

    # Plot learning curves
    plt.figure(figsize=(12, 4))
    plt.plot(results['lstm_history'].history['loss'], label='Training Loss')
    plt.plot(results['lstm_history'].history['val_loss'], label='Validation Loss')
    plt.title('LSTM Model Learning Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    return results

# Execute the analysis
results = main(df)

# Print final results and insights
print("\nFinal Results Summary:")
print("----------------------")
print("Best performing model:", results['metrics'].iloc[results['metrics']['MSE'].argmin()]['Model'])
print("\nDetailed metrics for each model:")
print(results['metrics'])

Data shape: (214152, 16)

First few rows:
               time  temperature  relative_humidity  dew_point  \
0  2000-01-01T00:00         -0.1                 97       -0.5   
1  2000-01-01T01:00         -0.2                 97       -0.7   
2  2000-01-01T02:00         -0.1                 96       -0.6   
3  2000-01-01T03:00         -0.2                 96       -0.7   
4  2000-01-01T04:00         -0.2                 96       -0.8   

   precipitation (mm)  rain (mm)  snowfall (cm)  pressure_msl (hPa)  \
0                 0.0        0.0            0.0              1024.2   
1                 0.0        0.0            0.0              1024.4   
2                 0.0        0.0            0.0              1024.5   
3                 0.0        0.0            0.0              1024.2   
4                 0.0        0.0            0.0              1024.0   

   surface_pressure (hPa)  cloud_cover (%)  cloud_cover_low (%)  \
0                  1019.3               87                   76   


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 176s 36ms/step - loss: 0.0048 - mae: 0.0402 - val_loss: 0.0038 - val_mae: 0.0508
Epoch 2/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 162s 34ms/step - loss: 3.3854e-04 - mae: 0.0141 - val_loss: 0.0047 - val_mae: 0.0583
Epoch 3/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 160s 33ms/step - loss: 1.9357e-04 - mae: 0.0105 - val_loss: 0.0059 - val_mae: 0.0669
Epoch 4/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 201s 33ms/step - loss: 1.3753e-04 - mae: 0.0088 - val_loss: 0.0064 - val_mae: 0.0705
Epoch 5/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 160s 33ms/step - loss: 1.2014e-04 - mae: 0.0082 - val_loss: 0.0064 - val_mae: 0.0706
Epoch 6/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 157s 33ms/step - loss: 1.0780e-04 - mae: 0.0077 - val_loss: 0.0076 - val_mae: 0.0764
Epoch 7/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 207s 34ms/step - loss: 9.7430e-05 - mae: 0.0073 - val_loss: 0.0066 - val_mae: 0.0719
Epoch 8/50
4818/4818 ━━━━━━━━━━━━━━━━━━━━ 199s 33ms/step - loss: 9.4119e-05 - mae: 0.0072 - val_loss: 0.0069 - val_

/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
